# 🧹 Data Cleaning — Database Q3 2020
> **Mục tiêu:** Clean 3 sheet chính dùng cho dashboard (MKT, Sales, Vận đơn)  
> **Input:** `Database_-_Q3_2020.xlsx`  
> **Output:** `Q3_2020_cleaned.xlsx` gồm 3 sheet đã clean

---
## 📋 Data Issues tìm thấy
| Sheet | Issues chính |
|---|---|
| **MKT** | Inbox dtype=object (mixed), ME1 có 14 nulls, Đơn hàng là float |
| **Sales** | 171 dòng null hoàn toàn · SĐT lưu dạng float · Close Date sai năm · `Type of Lead` viết tắt không chuẩn (Dathang/Tuvan) · `Sales Admin xác nhận` chỉ có 1 giá trị viết hoa DATTHANG · Tên cột tiếng Việt lẫn tiếng Anh không nhất quán |
| **Vận đơn** | 2,124 dòng null (duplicate product rows), nhiều cột không cần thiết |


## 0️⃣ Setup & Import

In [ ]:
# Nếu chạy trên Google Colab, upload file trước:
# from google.colab import files
# uploaded = files.upload()  # chọn file Database_-_Q3_2020.xlsx

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

FILE = 'Database_-_Q3_2020.xlsx'

print('✅ Libraries loaded')
print('📂 Sheets available:', pd.ExcelFile(FILE).sheet_names)

---
## 1️⃣ Sheet MKT — Marketing Performance

In [ ]:
# --- Load ---
mkt = pd.read_excel(FILE, sheet_name='MKT')
print(f'Raw shape: {mkt.shape}')
mkt.head(3)

In [ ]:
# --- Audit ---
print('=== DTYPES ===')
print(mkt.dtypes)
print('\n=== NULL COUNT ===')
print(mkt.isnull().sum())
print('\n=== INBOX sample (mixed type) ===')
print(mkt['Inbox'].value_counts().head(10))

In [ ]:
# ============================================================
# CLEAN MKT
# ============================================================
df_mkt = mkt.copy()

# 1. Rename columns — bỏ ký tự đặc biệt và xuống dòng
df_mkt.columns = [
    'Date', 'Channel', 'MKTer', 'Campaign',
    'Spend', 'Impression', 'Reach', 'Click',
    'Share', 'Comment', 'Inbox', 'Lead',
    'Order', 'Revenue', 'Paid_Revenue_1',
    'CPL', 'Order_per_Lead', 'CPM', 'CPC',
    'Cost_per_Mess', 'ME1'
]
print('✅ Columns renamed')

# 2. Date — đảm bảo datetime
df_mkt['Date'] = pd.to_datetime(df_mkt['Date'], errors='coerce')
print(f'✅ Date range: {df_mkt["Date"].min().date()} → {df_mkt["Date"].max().date()}')

# 3. Inbox — có giá trị 'Cộng tác viên', số, NaN → coerce về numeric
df_mkt['Inbox'] = pd.to_numeric(df_mkt['Inbox'], errors='coerce').fillna(0).astype(int)
print(f'✅ Inbox cleaned — unique non-zero: {df_mkt["Inbox"].nunique()}')

# 4. Order — float → int (không có lý do để là float)
df_mkt['Order'] = df_mkt['Order'].fillna(0).astype(int)
print('✅ Order → int')

# 5. ME1 — 14 nulls → fill bằng 0 (ngày không có data)
null_me1 = df_mkt['ME1'].isnull().sum()
df_mkt['ME1'] = df_mkt['ME1'].fillna(0)
print(f'✅ ME1: filled {null_me1} nulls → 0')

# 6. Thêm cột Month, Week để dùng trong dashboard
df_mkt['Month'] = df_mkt['Date'].dt.to_period('M').astype(str)
df_mkt['Week'] = df_mkt['Date'].dt.to_period('W').astype(str)
print('✅ Month & Week columns added')

# 7. Tính lại ROAS (Revenue / Spend) — tránh division by zero
df_mkt['ROAS'] = np.where(
    df_mkt['Spend'] > 0,
    (df_mkt['Revenue'] / df_mkt['Spend']).round(2),
    0
)
print('✅ ROAS column recalculated')

# 8. CTR
df_mkt['CTR'] = np.where(
    df_mkt['Impression'] > 0,
    (df_mkt['Click'] / df_mkt['Impression'] * 100).round(4),
    0
)
print('✅ CTR column added')

print(f'\n📊 MKT clean shape: {df_mkt.shape}')
df_mkt.head(3)

In [ ]:
# --- Verify MKT ---
print('=== MKT SUMMARY ===')
print(f'Total spend:   {df_mkt["Spend"].sum():>15,.0f} VND')
print(f'Total revenue: {df_mkt["Revenue"].sum():>15,.0f} VND')
print(f'Overall ROAS:  {df_mkt["Revenue"].sum() / df_mkt["Spend"].sum():>14.2f}x')
print(f'Total leads:   {df_mkt["Lead"].sum():>15,.0f}')
print(f'Total orders:  {df_mkt["Order"].sum():>15,.0f}')
print(f'Nulls left:    {df_mkt.isnull().sum().sum()}')

---
## 2️⃣ Sheet Sales — Lead & Conversion Data

In [ ]:
# --- Load ---
sales = pd.read_excel(FILE, sheet_name='Sales')
print(f'Raw shape: {sales.shape}')
sales.head(3)

In [ ]:
# --- Audit Sales ---
print('=== NULL COUNTS ===')
null_pct = (sales.isnull().sum() / len(sales) * 100).round(1)
for col, pct in null_pct[null_pct > 0].items():
    print(f'  {col:<45} {pct:>5}% null')

print('\n=== [1] TYPE OF LEAD — viết tắt không chuẩn ===')
print(sales['Type of Lead'].value_counts(dropna=False))
# Dathang  → Đặt hàng
# Tuvan    → Tư vấn
# (1 dòng) 'Con đang học lớp 7' → sai, cần flag

print('\n=== [2] SALES ADMIN XÁC NHẬN — viết hoa không chuẩn ===')
print(sales['Sales Admin xác nhận Type of Lead'].value_counts(dropna=False))
# DATTHANG → Đặt hàng (đồng nhất với Type of Lead)

print('\n=== [3] TÊN CỘT chưa chuẩn ===')
for i, col in enumerate(sales.columns):
    print(f'  [{i:>2}] {repr(col)}')

print('\n=== [4] CLOSE DATE year distribution ===')
print(pd.to_datetime(sales['Close Date'], errors='coerce').dt.year.value_counts())

print('\n=== [5] SĐT sample (stored as float) ===')
print(sales['SĐT'].head(5).tolist())

print('\n=== [6] CHANNEL — nguồn nào xuất hiện ===')
print(sales['Channel'].value_counts(dropna=False))

In [ ]:
# ============================================================
# CLEAN SALES — bao gồm chuẩn hóa giá trị viết tắt & tên cột
# ============================================================
df_sales = sales.copy()

# ── BƯỚC 1: Drop 171 dòng NULL hoàn toàn ────────────────────
before = len(df_sales)
df_sales = df_sales.dropna(subset=['Unnamed: 0', 'Khách hàng', 'SĐT', 'Channel', 'Sales'])
print(f'✅ [1] Dropped {before - len(df_sales)} fully-null rows | Remaining: {len(df_sales)}')

# ── BƯỚC 2: Rename cột — chuẩn hóa toàn bộ sang snake_case nhất quán ──
# Nguyên tắc: tiếng Anh, snake_case, rõ nghĩa, không dấu
df_sales.columns = [
    'lead_id',            # Unnamed: 0  → mã lead theo ngày
    'lead_time',          # Giờ         → giờ nhận lead
    'customer_name',      # Khách hàng
    'phone',              # SĐT
    'channel',            # Channel
    'campaign',           # Chiến dịch
    'ad_content',         # Content     → nội dung quảng cáo
    'marketer',           # Marketer 2
    'lead_type',          # Type of Lead
    'lead_type_confirmed',# Sales Admin xác nhận Type of Lead
    'salesperson',        # Sales
    'interaction_count',  # Số lần tương tác
    'call_date',          # Ngày gọi
    'status',             # Trạng thái
    'level',              # Level
    'followup_date',      # Ngày hẹn gọi lại
    'close_date',         # Close Date
    'province',           # Tỉnh/TP
    'book_qty',           # Số lượng bộ sách
    'discount',           # Số tiền giảm giá
    'total_amount'        # Tổng tiền
]
print('✅ [2] Columns renamed → snake_case nhất quán')
print('   Mapping nổi bật:')
print('     Unnamed: 0                          → lead_id')
print('     Type of Lead                        → lead_type')
print('     Sales Admin xác nhận Type of Lead   → lead_type_confirmed')
print('     Marketer 2                          → marketer')
print('     Tỉnh/TP                             → province')
print('     Content                             → ad_content')

# ── BƯỚC 3: Chuẩn hóa lead_type — viết tắt → full text ─────
# Giá trị gốc: 'Dathang', 'Tuvan', 'Con đang học lớp 7' (data lỗi)
print(f'\n📋 [3] lead_type trước khi clean:')
print(df_sales['lead_type'].value_counts(dropna=False))

lead_type_map = {
    'Dathang'              : 'Đặt hàng',   # viết tắt không dấu → full
    'Tuvan'                : 'Tư vấn',     # viết tắt không dấu → full
    'Con đang học lớp 7'   : None,         # data lỗi (câu trả lời khách), gán null
}
df_sales['lead_type'] = df_sales['lead_type'].map(
    lambda x: lead_type_map.get(x, x) if pd.notna(x) else x
)
print(f'\n✅ lead_type sau khi clean:')
print(df_sales['lead_type'].value_counts(dropna=False))

# ── BƯỚC 4: Chuẩn hóa lead_type_confirmed ───────────────────
# Giá trị gốc: 'DATTHANG' (viết hoa toàn bộ, viết tắt)
# → đồng nhất với lead_type: 'Đặt hàng'
print(f'\n📋 [4] lead_type_confirmed trước khi clean:')
print(df_sales['lead_type_confirmed'].value_counts(dropna=False))

confirmed_map = {
    'DATTHANG': 'Đặt hàng',   # ALLCAPS viết tắt → full text, đồng nhất với lead_type
}
df_sales['lead_type_confirmed'] = df_sales['lead_type_confirmed'].map(
    lambda x: confirmed_map.get(str(x).strip(), x) if pd.notna(x) else x
)
print(f'\n✅ lead_type_confirmed sau khi clean:')
print(df_sales['lead_type_confirmed'].value_counts(dropna=False))

# ── BƯỚC 5: Phone — float → string ──────────────────────────
df_sales['phone'] = df_sales['phone'].apply(
    lambda x: str(int(x)) if pd.notna(x) and x != 0 else None
)
print('\n✅ [5] phone cleaned: float → string')

# ── BƯỚC 6: Dates ────────────────────────────────────────────
# close_date — sửa năm sai (2023 → 2020, do Excel date serialization)
df_sales['close_date'] = pd.to_datetime(df_sales['close_date'], errors='coerce')
mask_wrong_year = df_sales['close_date'].dt.year.isin([2023, 2024, 2025])
n_wrong = mask_wrong_year.sum()
df_sales.loc[mask_wrong_year, 'close_date'] = (
    df_sales.loc[mask_wrong_year, 'close_date']
    .apply(lambda x: x.replace(year=2020) if pd.notna(x) else x)
)
print(f'✅ [6] close_date: fixed {n_wrong} wrong-year entries → 2020')
df_sales['call_date'] = pd.to_datetime(df_sales['call_date'], errors='coerce')
df_sales['followup_date'] = pd.to_datetime(df_sales['followup_date'], errors='coerce')
print('✅ [6] call_date, followup_date → datetime')

# ── BƯỚC 7: Thêm status_group (gom nhóm funnel) ─────────────
status_map = {
    'Đặt hàng'                 : 'Chuyển đổi',
    'Hủy đơn sau khi chốt'     : 'Hủy',
    'Từ chối'                  : 'Từ chối',
    'Suy nghĩ thêm/Trả lời sau': 'Đang xử lý',
    'Hẹn lúc khác gọi lại'     : 'Đang xử lý',
    'Không nghe máy'           : 'Không liên lạc được',
    'Thuê bao'                 : 'Không liên lạc được',
    'Sai số'                   : 'Không hợp lệ',
    'Trùng'                    : 'Không hợp lệ',
    'Trêu đùa'                 : 'Không hợp lệ',
}
df_sales['status'] = df_sales['status'].str.strip()
df_sales['status_group'] = df_sales['status'].map(status_map).fillna('Chưa phân loại')
print('✅ [7] status_group column added (tiếng Việt, 6 nhóm)')

# ── BƯỚC 8: Numeric cleanup ──────────────────────────────────
df_sales['book_qty']  = df_sales['book_qty'].fillna(0).astype(int)
df_sales['discount']  = df_sales['discount'].fillna(0)
df_sales['province']  = df_sales['province'].str.strip()
df_sales['level']     = df_sales['level'].str.strip()
print('✅ [8] book_qty, discount nulls → 0 | province/level stripped')

print(f'\n📊 Sales clean shape: {df_sales.shape}')
df_sales[['lead_type','lead_type_confirmed','status','status_group','channel']].head(10)

In [ ]:
# --- Verify Sales ---
print('=== SALES SUMMARY ===')
print(f'Total leads:  {len(df_sales):>10,}')

print('\n── lead_type (sau chuẩn hóa) ──')
print(df_sales['lead_type'].value_counts(dropna=False))

print('\n── lead_type_confirmed (sau chuẩn hóa) ──')
print(df_sales['lead_type_confirmed'].value_counts(dropna=False))

print('\n── status_group breakdown ──')
sg = df_sales['status_group'].value_counts()
for k, v in sg.items():
    print(f'  {k:<25} {v:>6,}  ({v/len(df_sales)*100:.1f}%)')

print(f'\nConversion rate: {(df_sales["status_group"]=="Chuyển đổi").mean()*100:.1f}%')

print('\n── close_date year after fix ──')
print(df_sales['close_date'].dt.year.value_counts())

print('\n── Nulls in key columns ──')
key_cols = ['lead_id','customer_name','channel','campaign','lead_type','lead_type_confirmed','status','status_group']
print(df_sales[key_cols].isnull().sum())

---
## 3️⃣ Sheet Vận đơn — Orders & Delivery

In [ ]:
# --- Load ---
orders = pd.read_excel(FILE, sheet_name='Vận đơn')
print(f'Raw shape: {orders.shape}')
orders.head(4)

In [ ]:
# --- Audit Vận đơn ---
print('=== NULL % per column ===')
null_pct = (orders.isnull().sum() / len(orders) * 100).round(1)
for col, pct in null_pct[null_pct > 0].items():
    print(f'  {col:<35} {pct:>6}%')

print('\n=== Delivery status ===')
print(orders['Tình trạng gói hàng'].value_counts())

print('\n=== Delivery partner ===')
print(orders['Đối tác giao hàng'].value_counts())

In [ ]:
# ============================================================
# CLEAN VẬN ĐƠN
# ============================================================
df_orders = orders.copy()

# 1. Xác định dòng "header" của order (có STT) vs dòng product detail
# Mỗi đơn hàng có thể có nhiều dòng sản phẩm. Dòng đầu tiên có STT là header.
# Giữ cả hai loại nhưng đánh dấu rõ
df_orders['Is_Order_Header'] = df_orders['STT'].notna()
print(f'Order headers (unique orders): {df_orders["Is_Order_Header"].sum():,}')
print(f'Product detail rows:           {(~df_orders["Is_Order_Header"]).sum():,}')

# 2. Tạo df_orders_main — chỉ lấy dòng order header (dùng cho dashboard level)
df_orders_main = df_orders[df_orders['Is_Order_Header']].copy()
print(f'\n✅ Main orders extracted: {len(df_orders_main)} rows')

# 3. Rename các cột quan trọng
rename_map = {
    'STT': 'Order_No',
    'Mã đơn hàng': 'Order_ID',
    'Tình trạng gói hàng': 'Delivery_Status',
    'Trạng thái đối tác': 'Partner_Status',
    'Ngày đóng gói': 'Pack_Date',
    'Ngày xuất kho': 'Dispatch_Date',
    'Ngày giao hàng': 'Delivery_Date',
    'Đối tác giao hàng': 'Courier',
    'Dịch vụ giao hàng': 'Courier_Service',
    'Tên người nhận': 'Recipient_Name',
    'SĐT người nhận': 'Recipient_Phone',
    'Tỉnh/Thành': 'Province',
    'Quận/Huyện': 'District',
    'Tiền khách phải trả cho đơn': 'Amount_Due',
    'Khách hàng đã trả': 'Amount_Paid',
    'Hình thức thanh toán': 'Payment_Method',
    'Phí vận chuyển': 'Shipping_Fee',
    'Tổng tiền hàng': 'Total_Amount',
    'Số lượng': 'Qty',
    'Đơn giá': 'Unit_Price'
}
df_orders_main = df_orders_main.rename(columns=rename_map)
print('✅ Columns renamed')

# 4. Dates → datetime
for col in ['Pack_Date', 'Dispatch_Date', 'Delivery_Date']:
    if col in df_orders_main.columns:
        df_orders_main[col] = pd.to_datetime(df_orders_main[col], errors='coerce')
print('✅ Date columns → datetime')

# 5. Tính Fulfillment_Days (xuất kho → giao hàng)
df_orders_main['Fulfillment_Days'] = (
    df_orders_main['Delivery_Date'] - df_orders_main['Dispatch_Date']
).dt.days
print(f'✅ Fulfillment_Days added | avg: {df_orders_main["Fulfillment_Days"].mean():.1f} days')

# 6. Delivery_Status — chuẩn hóa
df_orders_main['Delivery_Status'] = df_orders_main['Delivery_Status'].str.strip()

# 7. Thêm Delivery_Success flag
df_orders_main['Delivery_Success'] = df_orders_main['Delivery_Status'] == 'Đã giao hàng'

# 8. Province — strip
df_orders_main['Province'] = df_orders_main['Province'].str.strip()

# 9. Chỉ giữ cột cần thiết cho dashboard
keep_cols = [
    'Order_No', 'Order_ID', 'Delivery_Status', 'Partner_Status',
    'Pack_Date', 'Dispatch_Date', 'Delivery_Date', 'Fulfillment_Days',
    'Courier', 'Courier_Service', 'Province', 'District',
    'Amount_Due', 'Amount_Paid', 'Payment_Method', 'Shipping_Fee',
    'Total_Amount', 'Qty', 'Unit_Price', 'Delivery_Success'
]
# Chỉ giữ cột tồn tại
keep_cols = [c for c in keep_cols if c in df_orders_main.columns]
df_orders_main = df_orders_main[keep_cols]

print(f'\n📊 Orders clean shape: {df_orders_main.shape}')
df_orders_main.head(3)

In [ ]:
# --- Verify Orders ---
print('=== ORDERS SUMMARY ===')
print(f'Total orders:         {len(df_orders_main):>8,}')
print(f'Delivered:            {df_orders_main["Delivery_Success"].sum():>8,} ({df_orders_main["Delivery_Success"].mean()*100:.1f}%)')
print(f'\nDelivery status breakdown:')
print(df_orders_main['Delivery_Status'].value_counts())
print(f'\nAvg fulfillment days: {df_orders_main["Fulfillment_Days"].mean():.1f}')
print(f'Top 3 provinces:')
print(df_orders_main['Province'].value_counts().head(3))

---
## 4️⃣ Export — Lưu file đã clean

In [ ]:
OUTPUT = 'Q3_2020_cleaned.xlsx'

with pd.ExcelWriter(OUTPUT, engine='openpyxl') as writer:
    df_mkt.to_excel(writer, sheet_name='MKT_clean', index=False)
    df_sales.to_excel(writer, sheet_name='Sales_clean', index=False)
    df_orders_main.to_excel(writer, sheet_name='Orders_clean', index=False)

print(f'✅ Exported: {OUTPUT}')
print(f'   MKT_clean:    {df_mkt.shape}')
print(f'   Sales_clean:  {df_sales.shape}')
print(f'   Orders_clean: {df_orders_main.shape}')

In [ ]:
# Download file (chạy trên Google Colab)
# from google.colab import files
# files.download(OUTPUT)
print('📥 Uncomment 2 dòng trên nếu đang dùng Google Colab để download file')

---
## 5️⃣ Quick Sanity Check — Cross-sheet Validation

In [ ]:
print('='*55)
print('CROSS-SHEET SANITY CHECK')
print('='*55)

# Leads so sánh
leads_mkt = df_mkt['Lead'].sum()
leads_sales = len(df_sales)
print(f'\n📌 Leads:')
print(f'  MKT sheet reported:  {leads_mkt:>8,.0f}')
print(f'  Sales sheet rows:    {leads_sales:>8,}')
print(f'  Gap:                 {leads_sales - leads_mkt:>+8,.0f} ({(leads_sales-leads_mkt)/leads_mkt*100:+.1f}%)')
print('  → Bình thường (Sales ghi thêm leads organic / referral không qua MKT)')

# Orders so sánh
orders_mkt = df_mkt['Order'].sum()
orders_sales = (df_sales['status_group'] == 'Chuyển đổi').sum()
orders_system = len(df_orders_main)
print(f'\n📌 Orders:')
print(f'  MKT sheet reported:  {orders_mkt:>8,.0f}')
print(f'  Sales converted:     {orders_sales:>8,}')
print(f'  Order system (Vận đơn): {orders_system:>5,}')
print('  → Vận đơn lớn hơn vì có đơn từ nguồn khác ngoài MKT tracked')

# Revenue so sánh
rev_mkt = df_mkt['Revenue'].sum()
rev_orders = df_orders_main['Amount_Due'].sum()
print(f'\n📌 Revenue:')
print(f'  MKT attributed:  {rev_mkt:>15,.0f} VND')
print(f'  Orders amount:   {rev_orders:>15,.0f} VND')

print('\n✅ Sanity check complete — data sẵn sàng cho dashboard!')

---
## 📝 Tóm tắt thao tác đã làm

| Sheet | Thao tác |
|---|---|
| **MKT** | Rename 21 cột · Fix Inbox dtype (object→int) · Fill ME1 nulls · Fix Order (float→int) · Thêm Month/Week/ROAS/CTR |
| **Sales** | Drop 171 null rows · Rename 21 cột → snake_case · `lead_type`: Dathang→Đặt hàng, Tuvan→Tư vấn, data lỗi→null · `lead_type_confirmed`: DATTHANG→Đặt hàng · Phone float→string · Fix close_date year 2023→2020 · Thêm status_group (6 nhóm tiếng Việt) · Fill book_qty/discount |
| **Vận đơn** | Tách header rows vs product rows · Rename cột · Parse dates · Tính Fulfillment_Days · Thêm Delivery_Success flag · Giữ 20 cột cần thiết |

> **File output:** `Q3_2020_cleaned.xlsx` — 3 sheets sạch, sẵn sàng kéo vào Power BI / Looker Studio / Tableau.
